In [ ]:
import pandas as pd 
from pace_pic_validation.io.preprocessing import extract_start_date_from_nc
import glob
from pathlib import Path
import xarray as xr
from pace_pic_validation.viz.RGB_functions import rgb_plot_with_stations
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
from matplotlib.ticker import AutoMinorLocator
from pace_pic_validation.viz.visualization import plot_rrs_spectra
import re
import os
import pickle
from scipy.spatial import cKDTree
from sklearn.neighbors import BallTree, KDTree
from pace_pic_validation.io.DALEC import openDALEC
from pace_pic_validation.matchup.processing import CL2algorithm


In [ ]:
path_merge_ds = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/merged_datasets'
path_all_pic  = '/Volumes/Extreme SSD/PhD backup/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/PIC_discrete/all_PIC.csv'

SS20260602_allwls = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/merged_datasets/all_wls/2025-06-01_2025-06-06_latmin_50.0_latmax_57.0_lonmin_-1.0_lonmax_7.0_[CLDICE].nc'
SS20250601_allwls = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/merged_datasets/all_wls/2025-06-01_2025-06-06_latmin_50.0_latmax_57.0_lonmin_-1.0_lonmax_7.0_[CLDICE].nc'

SS20260601_DALEC_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/DALEC/ProcDalec_L2_202606_M99.pkl'
SS20250601_DALEC_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/DALEC/ProcDalec_L2_202506_M99.pkl'



In [ ]:

# seperating stations of all_pic

df = pd.read_csv(path_all_pic)
df = df.sort_values('datetime')

groups = {k: v.copy() for k, v in df.groupby("Campaign")}

SS_campaigns= {k: v for k, v in groups.items() if k.startswith("SS")}

print('available Campaigns')
print(groups.keys())

print('available Simon Stevins Campaigns')
print(SS_campaigns.keys())

# sorting merged datasets
paths_merged_ds = glob.glob(path_merge_ds + '/*.nc')
paths_sorted = sorted(paths_merged_ds)




In [ ]:

# Map from date string (YYYYMMDD) -> nc file path (or loaded dataset)
date_to_nc = {}
for f in paths_sorted:
    d = extract_start_date_from_nc(f)
    if d is not None:
        date_to_nc[d] = f  # or xr.open_dataset(f) if you want to load now

# Now match with ss_dict keys
matched = {}
for key, ds in SS_campaigns.items():
    # key is like "SS20250602"
    date_part = key[2:]  # "20250602"
    if date_part in date_to_nc:
        matched[key] = (ds, date_to_nc[date_part])
    else:
        matched[key] = (ds, None)  # or handle missing case


In [ ]:

df = SS_campaigns['SS20260601']
df = df.where(df['Station'].str.endswith(('02', '03', '04', '05', '06', '07', '09', '10', '11')))
df = df.dropna(axis = 'rows')
df = df.reset_index()


lons = list(df["lon"])
lats = list(df["lat"])
stations = list(df["Station"])
datetimes = list(df["datetime"])

print(datetimes[0])

SS20260601_allwls = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/merged_datasets/all_wls/2026-06-01_2026-06-05_latmin_50.0_latmax_57.0_lonmin_-1.0_lonmax_7.0_[CLDICE].nc'
SS20250601_allwls = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/merged_datasets/all_wls/2025-06-01_2025-06-06_latmin_50.0_latmax_57.0_lonmin_-1.0_lonmax_7.0_[CLDICE].nc'

SS20260601_DALEC_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/DALEC/ProcDalec_L2_202606_M99.pkl'
SS20260602_DALEC_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/DALEC/ProcDalec_L2_202506_M99.pkl'

ds = xr.open_dataset(SS20260601_allwls)

save_dir = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Figures/Ocean Optics/'


In [ ]:


# DALEC processing

DALEC_paths = [SS20250601_DALEC_path, SS20260601_DALEC_path]
keys = ['SS202506', 'SS202606']

# dict 
DALEC_ds = {}


for DALEC_path, key in zip(DALEC_paths, keys): 

    df = openDALEC(DALEC_path)

    DALEC_ds[key] = df


def chl_oc6_dalec(rrs):
        """
        rrs: dict-like with keys at or near:
            412, 443, 490, 510, 555, 678 (nm)
        returns chlorophyll-a in mg m-3 using OC6_PACE_OCI.
        """
        # rrs is a DataFrame, columns include 412, 443, 490, 510, 555, 678
        blue_cols = [412, 444, 490, 510]
        green_cols = [556, 678]

        rrs_blue_max = rrs[blue_cols].max(axis=1)          # Series, one value per row
        rrs_green_mean = rrs[green_cols].mean(axis=1)      # Series, one value per row


        MBR = rrs_blue_max / rrs_green_mean
        X = np.log10(MBR)

        a0, a1, a2, a3, a4 = 0.94297, -3.18493, 2.33682, -1.23923, 0.18697

        log_chl = a0 + a1*X + a2*X**2 + a3*X**3 + a4*X**4
        return 10**log_chl


print(df.shape)
df['chla_mg_m3'] = chl_oc6_dalec(df)

print(df['chla_mg_m3'])

In [ ]:
df = SS_campaigns['SS20260601']

print(df.columns)

In [ ]:
discrete_dir_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/matchups_discrete'
lisstpic_dir_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/matchups_lisstpic'


paths = [discrete_dir_path, lisstpic_dir_path]

discrete_list = [os.path.join(path, f) for path in paths for f in os.listdir(path) if f.endswith('.csv')]
lisstpic_list = [os.path.join(path, f) for path in paths for f in os.listdir(path) if f.endswith('.csv')]


discrete = pd.concat((pd.read_csv(f) for f in discrete_list))
lisstpic = pd.concat((pd.read_csv(f) for f in lisstpic_list))

print(len(discrete['granule'].unique()))
discrete = discrete.dropna(subset ='Rrs_547_px4_raw')
print(len(discrete['granule'].unique()))




In [ ]:
# function to plot rrs spectra from PACE L2

granule_path ='/Volumes/Extreme SSD/PhD backup/Satellite Data/PACE_L2/PACE_OCI.20240602T123712.L2.OC_AOP.V3_2.nc'

ds1 = xr.open_dataset(SS20260601_allwls)
ds2 = xr.open_dataset(SS20250601_allwls)

def chl_oc6_pace_granule(granule_path, lon, lat):
    """
    Compute a single chlorophyll-a value for a given (lon, lat)
    from a PACE OCI L2 granule using the OC6_PACE_OCI algorithm.

    Parameters
    ----------
    granule_path : str
        Path to the PACE L2 granule file.
    lon, lat : float
        Target longitude and latitude.

    Returns
    -------
    chl : float
        Chlorophyll-a concentration in mg m-3.
    """
    from pace_pic_validation.io.preprocessing import open_l2_minimal


    ds = open_l2_minimal(granule_path)


    # Distance-based nearest pixel (great-circle approximation)
    dlat = ds.latitude - lat
    dlon = ds.longitude - lon
    dist = np.sqrt(dlat**2 + (dlon * np.cos(np.deg2rad(lat)))**2)

    idx = dist.argmin(dim=["number_of_lines", "pixels_per_line"])

    ds_pt = ds.isel(
        number_of_lines=idx["number_of_lines"],
        pixels_per_line=idx["pixels_per_line"],
    )

    # Extract Rrs spectrum at this pixel: 1D array over wavelength
    rrs_spec = ds_pt["Rrs"]  # dims: (wavelength,)

    # Select blue and green bands (use nearest wavelengths)
    blue_wls = [412.0, 443.0, 490.0, 510.0]
    green_wls = [555.0, 678.0]

    rrs_blue = rrs_spec.sel(wavelength=blue_wls, method="nearest")
    rrs_green = rrs_spec.sel(wavelength=green_wls, method="nearest")

    # OC6_PACE_OCI: max over blue, mean over green
    rrs_blue_max = float(rrs_blue.max())
    rrs_green_mean = float(rrs_green.mean())

    if rrs_green_mean <= 0:
        return np.nan

    MBR = rrs_blue_max / rrs_green_mean
    X = np.log10(MBR)

    a0, a1, a2, a3, a4 = 0.94297, -3.18493, 2.33682, -1.23923, 0.18697

    log_chl = a0 + a1*X + a2*X**2 + a3*X**3 + a4*X**4
    chl = 10**log_chl

    return chl



chlA = chl_oc6_pace_granule(granule_path, 3.7, 53.677)
print(chlA)

def chl_oc6_pace_merged(merged_path, lat, lon):

    ds = xr.open_dataset(merged_path)

    lats = ds["latitude"]
    lons = ds["longitude"]

    dlat = lats - lat
    dlon = lons - lon
    dist = np.sqrt(
        dlat**2 + (dlon * np.cos(np.deg2rad(lat)))**2
    )

    # Stack lat & lon into a single dimension
    stacked = dist.stack(loc=("latitude", "longitude"))

    # Flat index of minimum distance
    min_idx = int(stacked.argmin(dim="loc").values)

    # Get corresponding lat/lon indices via unravel
    n_lat = ds.sizes["latitude"]
    n_lon = ds.sizes["longitude"]

    idx_lat, idx_lon = np.unravel_index(min_idx, (n_lat, n_lon))

    # Extract the nearest pixel
    ds_pt = ds.isel(latitude=idx_lat, longitude=idx_lon)

    blue_wls = [412.0, 443.0, 490.0, 510.0]
    green_wls = [555.0, 678.0]

    rrs_spec = ds_pt["Rrs"]  # 1D in wavelength

    rrs_blue = rrs_spec.sel(wavelength=blue_wls, method="nearest")
    rrs_green = rrs_spec.sel(wavelength=green_wls, method="nearest")
    # OC6_PACE_OCI: max over blue, mean over green

# OC6_PACE_OCI: max over blue, mean over green
    rrs_blue_max = float(rrs_blue.max())
    rrs_green_mean = float(rrs_green.mean())

    if rrs_green_mean <= 0:
        return np.nan

    MBR = rrs_blue_max / rrs_green_mean
    X = np.log10(MBR)

    a0, a1, a2, a3, a4 = 0.94297, -3.18493, 2.33682, -1.23923, 0.18697

    log_chl = a0 + a1*X + a2*X**2 + a3*X**3 + a4*X**4
    chl = 10**log_chl

    return chl



chla =chl_oc6_pace_merged(SS20260601_allwls, lon =3.7, lat =53.677)
print('does it work')
print(chla)



def pace_granule_plot_prep(granule_path, lon, lat):

    from pace_pic_validation.io.preprocessing import open_l2_minimal
    from pace_pic_validation.matchup.matchup_function import decode_l2_flags
    import matplotlib.pyplot as plt 

    ds = open_l2_minimal(granule_path)
 
    dlat = ds.latitude - lat
    dlon = ds.longitude - lon

    dist = np.sqrt(dlat**2 + (dlon * np.cos(np.deg2rad(lat)))**2)
    dist_km = dist * (np.pi / 180.0) * 6378 

    idx = dist.argmin(dim=["number_of_lines", "pixels_per_line"])

    ds_pt = ds.isel(
    number_of_lines=idx["number_of_lines"],
    pixels_per_line=idx["pixels_per_line"],
    )
    flag_int = [ds_pt['l2_flags'].values]
    lat_pace = ds_pt['latitude'].values
    lon_pace = ds_pt['longitude'].values
    flags_list = decode_l2_flags(flag_int)

    print(f'{flags_list} for spectrum at {lat_pace}, {lon_pace}, distance to desired lat lon is {dist_km.min().item()} km')
    x = ds_pt['wavelength']
    y = ds_pt['Rrs']

    return x, y, flags_list


x,y,flag_list = pace_granule_plot_prep(granule_path, lon = 6.10, lat = 10)


plt.plot(x, y)


plt.show()

In [ ]:
matchups_discrete_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/matchups_discrete/240min_2km/matchups_2024-05-21 23:56:15_2026-06-05 16:03:35.csv'
matchups_discrete = pd.read_csv(matchups_discrete_path)

matchups_discrete['ci2_pic'] = CL2algorithm(matchups_discrete['Rrs_547_px4_raw'], matchups_discrete['Rrs_667_px4_raw'], old = True) *1000


print(matchups_discrete.columns)



In [ ]:
from typing import Sequence, Optional, Union, List, Tuple
import numpy as np
def plot_rrs_spectra(
    ds: xr.Dataset,
    lons: Sequence[float],
    lats: Sequence[float],
    stations: Sequence[str],
    rrs_variable: str = "Rrs",
    wavelength_variable: str = "wavelength",
    mode: str = "multi",  # "single" or "multi"
    figsize: Tuple[float, float] = (6.5, 4.5),
    dpi: int = 150,
    xlim: Optional[Tuple[float, float]] = (400, 700),
    ylim: Optional[Tuple[float, float]] = None,
    xticks: Optional[Sequence[float]] = None,
    xlabel: str = "Wavelength (nm)",
    ylabel: str = r"Remote-sensing reflectance, $R_{rs}$ (sr$^{-1}$)",
    title: Optional[str] = None,
    legend_title: str = "Station",
    cmap: str = "viridis",
    cmap_range: Tuple[float, float] = (0, 0.9),
    line_kwargs: Optional[dict] = None,
    grid: bool = True,
    grid_kwargs: Optional[dict] = None,
    minor_ticks: bool = True,
    spines_top_right: bool = True,
    save_dir: Optional[str] = None,
    save_kwargs: Optional[dict] = None,
    show: bool = True,
    apply_style: bool = True,
    dalec = None,
    src_rrs = 'rgb', # 'rgb' or 'granule',
    matchup_ds = None, # 
    discrete_ds = None

) -> plt.Figure:
    """
    Plot Rrs spectra for one or multiple stations.

    Parameters
    ----------
    ds : xr.Dataset
        xarray Dataset containing the Rrs variable and wavelength coordinate.
    lons : sequence of float
        Longitudes for each station.
    lats : sequence of float
        Latitudes for each station.
    stations : sequence of str
        Station names/labels for each location.
    rrs_variable : str, optional
        Name of the Rrs variable in the Dataset.
    wavelength_variable : str, optional
        Name of the wavelength coordinate/variable.
    mode : {"single", "multi"}, optional
        "single" → create one figure per station (calls plt.show() each time).
        "multi" → plot all stations on a single figure.
    figsize : tuple, optional
        Figure size (width, height) in inches.
    dpi : int, optional
        Figure resolution.
    xlim : tuple or None, optional
        (xmin, xmax) for x-axis. Set to None to auto-scale.
    ylim : tuple or None, optional
        (ymin, ymax) for y-axis. Set to None to auto-scale.
    xticks : sequence or None, optional
        Tick locations on x-axis.
    xlabel : str, optional
        X-axis label.
    ylabel : str, optional
        Y-axis label.
    title : str or None, optional
        Figure title. If None, no title is added.
    legend_title : str, optional
        Title for the legend.
    cmap : str, optional
        Matplotlib colormap for station colors.
    cmap_range : tuple, optional
        Fractional range of colormap to use, e.g. (0, 0.9).
    line_kwargs : dict or None, optional
        Additional keyword arguments passed to `ax.plot()` for each spectrum.
        Example: {"linewidth": 0.6, "alpha": 0.85}.
    grid : bool, optional
        Whether to draw a grid.
    grid_kwargs : dict or None, optional
        Keyword arguments for `ax.grid()`, e.g. {"ls": ":", "alpha": 0.9}.
    minor_ticks : bool, optional
        Whether to add minor ticks on both axes.
    spines_top_right : bool, optional
        Whether to show top and right spines.
    save_path : str or None, optional
        Path (including filename) to save the figure. If None, figure is not saved.
    save_kwargs : dict or None, optional
        Additional keyword arguments for `fig.savefig()`, e.g. {"dpi": 300}.
    show : bool, optional
        Whether to call `plt.show()` at the end.
    apply_style : bool, optional
        Whether to apply a publication-style `rcParams` update.

    Returns
    -------
    fig : matplotlib.figure.Figure
        The created figure object.


    """
    from pace_pic_validation.io.DALEC import xy_dalec

    

    def chl_pic_oc6_pace_merged(ds, lat, lon):


        lats = ds["latitude"]
        lons = ds["longitude"]

        dlat = lats - lat
        dlon = lons - lon
        dist = np.sqrt(
            dlat**2 + (dlon * np.cos(np.deg2rad(lat)))**2
        )

        # Stack lat & lon into a single dimension
        stacked = dist.stack(loc=("latitude", "longitude"))

        # Flat index of minimum distance
        min_idx = int(stacked.argmin(dim="loc").values)

        # Get corresponding lat/lon indices via unravel
        n_lat = ds.sizes["latitude"]
        n_lon = ds.sizes["longitude"]

        idx_lat, idx_lon = np.unravel_index(min_idx, (n_lat, n_lon))

        # Extract the nearest pixel
        ds_pt = ds.isel(latitude=idx_lat, longitude=idx_lon)

        blue_wls = [412.0, 443.0, 490.0, 510.0]
        green_wls = [555.0, 678.0]

        rrs_spec = ds_pt["Rrs"]  # 1D in wavelength

        rrs_blue = rrs_spec.sel(wavelength=blue_wls, method="nearest")
        rrs_green = rrs_spec.sel(wavelength=green_wls, method="nearest")

        rrs_547 = rrs_spec.sel(wavelength=547, method="nearest")
        rrs_667 = rrs_spec.sel(wavelength=667, method="nearest")
        # OC6_PACE_OCI: max over blue, mean over green

    # OC6_PACE_OCI: max over blue, mean over green
        rrs_blue_max = float(rrs_blue.max())
        rrs_green_mean = float(rrs_green.mean())

        if rrs_green_mean <= 0:
            return np.nan

        MBR = rrs_blue_max / rrs_green_mean
        X = np.log10(MBR)

        a0, a1, a2, a3, a4 = 0.94297, -3.18493, 2.33682, -1.23923, 0.18697

        log_chl = a0 + a1*X + a2*X**2 + a3*X**3 + a4*X**4
        chl = 10**log_chl

        pic = CL2algorithm(rrs_547, rrs_667, old = True) *1000

        return chl, pic


    def chl_pic_oc6_pace_granule(granule_path, lon, lat):
        """
        Compute a single chlorophyll-a value for a given (lon, lat)
        from a PACE OCI L2 granule using the OC6_PACE_OCI algorithm.

        Parameters
        ----------
        granule_path : str
            Path to the PACE L2 granule file.
        lon, lat : float
            Target longitude and latitude.

        Returns
        -------
        chl : float
            Chlorophyll-a concentration in mg m-3.
        """
        from pace_pic_validation.io.preprocessing import open_l2_minimal


        ds = open_l2_minimal(granule_path)


        # Distance-based nearest pixel (great-circle approximation)
        dlat = ds.latitude - lat
        dlon = ds.longitude - lon
        dist = np.sqrt(dlat**2 + (dlon * np.cos(np.deg2rad(lat)))**2)

        idx = dist.argmin(dim=["number_of_lines", "pixels_per_line"])

        ds_pt = ds.isel(
            number_of_lines=idx["number_of_lines"],
            pixels_per_line=idx["pixels_per_line"],
        )

        # Extract Rrs spectrum at this pixel: 1D array over wavelength
        rrs_spec = ds_pt["Rrs"]  # dims: (wavelength,)

        # Select blue and green bands (use nearest wavelengths)
        blue_wls = [412.0, 443.0, 490.0, 510.0]
        green_wls = [555.0, 678.0]

        rrs_blue = rrs_spec.sel(wavelength=blue_wls, method="nearest")
        rrs_green = rrs_spec.sel(wavelength=green_wls, method="nearest")

        # OC6_PACE_OCI: max over blue, mean over green
        rrs_blue_max = float(rrs_blue.max())
        rrs_green_mean = float(rrs_green.mean())
        rrs_547 = rrs_spec.sel(wavelength=547, method="nearest")
        rrs_667 = rrs_spec.sel(wavelength=677, method="nearest")

        pic = CL2algorithm(rrs_547, rrs_667, old = True)

        if rrs_green_mean <= 0:
            return np.nan

        MBR = rrs_blue_max / rrs_green_mean
        X = np.log10(MBR)

        a0, a1, a2, a3, a4 = 0.94297, -3.18493, 2.33682, -1.23923, 0.18697

        log_chl = a0 + a1*X + a2*X**2 + a3*X**3 + a4*X**4
        chl = 10**log_chl

        return chl, pic


    def pace_granule_plot_prep(granule_path, lon, lat):

        from pace_pic_validation.io.preprocessing import open_l2_minimal
        from pace_pic_validation.matchup.matchup_function import decode_l2_flags
        import matplotlib.pyplot as plt 

        ds = open_l2_minimal(granule_path)
    
        dlat = ds.latitude - lat
        dlon = ds.longitude - lon

        dist = np.sqrt(dlat**2 + (dlon * np.cos(np.deg2rad(lat)))**2)
        dist_km = dist * (np.pi / 180.0) * 6378 

        idx = dist.argmin(dim=["number_of_lines", "pixels_per_line"])

        ds_pt = ds.isel(
        number_of_lines=idx["number_of_lines"],
        pixels_per_line=idx["pixels_per_line"],
        )
        flag_int = [ds_pt['l2_flags'].values]
        lat_pace = ds_pt['latitude'].values
        lon_pace = ds_pt['longitude'].values
        flags_list = decode_l2_flags(flag_int)

        print(f'{flags_list} for spectrum at {lat_pace}, {lon_pace}, distance to desired lat lon is {dist_km.min().item()} km')
        x = ds_pt['wavelength']
        y = ds_pt['Rrs']

        return x, y, flags_list


    def pace_granule_plot_prep(granule_path, lon, lat):

        from pace_pic_validation.io.preprocessing import open_l2_minimal
        from pace_pic_validation.matchup.matchup_function import decode_l2_flags
        import matplotlib.pyplot as plt 

        ds = open_l2_minimal(granule_path)
    
        dlat = ds.latitude - lat
        dlon = ds.longitude - lon

        dist = np.sqrt(dlat**2 + (dlon * np.cos(np.deg2rad(lat)))**2)
        dist_km = dist * (np.pi / 180.0) * 6378 

        idx = dist.argmin(dim=["number_of_lines", "pixels_per_line"])

        ds_pt = ds.isel(
        number_of_lines=idx["number_of_lines"],
        pixels_per_line=idx["pixels_per_line"],
        )
        flag_int = [ds_pt['l2_flags'].values]
        lat_pace = ds_pt['latitude'].values
        lon_pace = ds_pt['longitude'].values
        flags_list = decode_l2_flags(flag_int)

        print(f'{flags_list} for spectrum at {lat_pace}, {lon_pace}, distance to desired lat lon is {dist_km.min().item()} km')
        x = ds_pt['wavelength']
        y = ds_pt['Rrs']

        return x, y, flags_list

    def chl_oc6_dalec(rrs):
        """
        rrs: dict-like with keys at or near:
            412, 443, 490, 510, 555, 678 (nm)
        returns chlorophyll-a in mg m-3 using OC6_PACE_OCI.
        """
        # rrs is a DataFrame, columns include 412, 443, 490, 510, 555, 678
        blue_cols = [412, 444, 490, 510]
        green_cols = [556, 678]

        rrs_blue_max = rrs[blue_cols].max(axis=1)          # Series, one value per row
        rrs_green_mean = rrs[green_cols].mean(axis=1)      # Series, one value per row


        MBR = rrs_blue_max / rrs_green_mean
        X = np.log10(MBR)

        a0, a1, a2, a3, a4 = 0.94297, -3.18493, 2.33682, -1.23923, 0.18697

        log_chl = a0 + a1*X + a2*X**2 + a3*X**3 + a4*X**4
        return 10**log_chl

    def chl_oc6_pace_granule(granule_path, lon, lat):
        """
        Compute a single chlorophyll-a value for a given (lon, lat)
        from a PACE OCI L2 granule using the OC6_PACE_OCI algorithm.

        Parameters
        ----------
        granule_path : str
            Path to the PACE L2 granule file.
        lon, lat : float
            Target longitude and latitude.

        Returns
        -------
        chl : float
            Chlorophyll-a concentration in mg m-3.
        """
        from pace_pic_validation.io.preprocessing import open_l2_minimal

        ds = open_l2_minimal(granule_path)

        # Distance-based nearest pixel (great-circle approximation)
        dlat = ds.latitude - lat
        dlon = ds.longitude - lon
        dist = np.sqrt(dlat**2 + (dlon * np.cos(np.deg2rad(lat)))**2)

        idx = dist.argmin(dim=["number_of_lines", "pixels_per_line"])

        ds_pt = ds.isel(
            number_of_lines=idx["number_of_lines"],
            pixels_per_line=idx["pixels_per_line"],
        )

        # Extract Rrs spectrum at this pixel: 1D array over wavelength
        rrs_spec = ds_pt["Rrs"]  # dims: (wavelength,)

        # Select blue and green bands (use nearest wavelengths)
        blue_wls = [412.0, 443.0, 490.0, 510.0]
        green_wls = [555.0, 678.0]

        rrs_blue = rrs_spec.sel(wavelength=blue_wls, method="nearest")
        rrs_green = rrs_spec.sel(wavelength=green_wls, method="nearest")

        # OC6_PACE_OCI: max over blue, mean over green
        rrs_blue_max = float(rrs_blue.max())
        rrs_green_mean = float(rrs_green.mean())

        if rrs_green_mean <= 0:
            return np.nan

        MBR = rrs_blue_max / rrs_green_mean
        X = np.log10(MBR)

        a0, a1, a2, a3, a4 = 0.94297, -3.18493, 2.33682, -1.23923, 0.18697

        log_chl = a0 + a1*X + a2*X**2 + a3*X**3 + a4*X**4
        chl = 10**log_chl

        return chl

    
    dalec['chla_mg_m3'] = chl_oc6_dalec(dalec)
    dalec['pic_mmol_m3'] = CL2algorithm(dalec[548], dalec[668], old = True)


    if len(lons) != len(lats) or len(lons) != len(stations):
        raise ValueError("lons, lats, and stations must have the same length.")

    if line_kwargs is None:
        line_kwargs = {}

    if grid_kwargs is None:
        grid_kwargs = {"ls": ":", "alpha": 0.9}

    if save_kwargs is None:
        save_kwargs = {}

    if apply_style:
        plt.style.use("default")
        plt.rcParams.update(
            {
                "font.size": 11,
                "font.family": "sans-serif",
                "font.sans-serif": [
                    "Arial",
                    "DejaVu Sans",
                    "Liberation Sans",
                ],
                "axes.linewidth": 1.2,
                "axes.labelsize": 12,
                "axes.titlelocation": "left",
                "axes.titleweight": "normal",
                "xtick.major.size": 6,
                "xtick.major.width": 1.2,
                "xtick.minor.size": 3,
                "xtick.minor.width": 1.0,
                "ytick.major.size": 6,
                "ytick.major.width": 1.2,
                "ytick.minor.size": 3,
                "ytick.minor.width": 1.0,
                "xtick.direction": "in",
                "ytick.direction": "in",
                "xtick.top": True,
                "ytick.right": True,
                "legend.frameon": False,
                "legend.fontsize": 10,
            }
        )




    if mode == "single":
        # One figure per station

        if src_rrs == 'rgb':

            for lon, lat, station in zip(lons, lats, stations):

                chla_rgb, pic_rgb = chl_pic_oc6_pace_merged(ds, lat, lon)

                print(f'chla for rgb: {chla_rgb}')

                test = ds[rrs_variable].sel(
                    longitude=lon,
                    latitude=lat,
                    method="nearest",
                )

                fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

                ax.plot(
                    test[wavelength_variable].values,
                    test.values,
                    **line_kwargs,
                    label = 'from merged L2 (Cruise Period)',
                )

                

                if dalec is not None:

                    # or ':', '-.', etc.
                    x,y,index = xy_dalec(dalec, lat, lon)
                    print(f'x is {x}')
                    print(f'y is {y}')
                    print(f'index is {index}')
                    ax.plot(
                            x,
                            y,
                            color="black",           # or another fixed color
                            linestyle="--",          # explicit, in case line_kwargs doesn't set it
                            label="DALEC (Ship-Based)",           # if you want it in the legend
                        )
                    chla_dalec = dalec['chla_mg_m3'][index]
                    pic_dalec = dalec['pic_mmol_m3'][index] * 1000

                    print(f'{chla_dalec} mg/m3 found from DALEC rrs')
                    print(f'{pic_dalec} mmol/m3 found from DALEC rrs')
                    


                # PIC and Chl A values

                

                pic = discrete_ds.loc[
                                    (discrete_ds["lon"] == lon) & (discrete_ds["lat"] == lat),
                                    "PIC_mmolC_m3"
                                ]

                print(f'{pic} discrete found for {lon}, {lat}')

                pic_ci2 = matchup_ds.loc[
                                    (matchup_ds["lon_insitu"] == lon) & (matchup_ds["lat_insitu"] == lat),
                                    "ci2_pic"
                                ]
                
                print(f'{pic_ci2} discrete found for {lon}, {lat}')


                def to_scalar_or_none(x):
                    # Returns None if x is “missing”, else a float scalar
                    if isinstance(x, pd.Series):
                        if x.empty:
                            return None
                        val = x.iloc[0]
                        return None if pd.isna(val) else float(val)
                    # Assume scalar-like
                    if pd.isna(x):
                        return None
                    return float(x)

                chla_dalec = to_scalar_or_none(chla_dalec)
                pic_dalec = to_scalar_or_none(pic_dalec)
                pic_ci2_rgb = to_scalar_or_none(pic_rgb)
                pic_val = to_scalar_or_none(pic)
                pic_ci2_val = to_scalar_or_none(pic_ci2)
                chla_rgb = to_scalar_or_none(chla_rgb)

                parts = []

                if pic_val is not None and np.isfinite(pic_val):
                    parts.append(f"PIC (ICP, In-Situ): {pic_val:.3f} mmol/m³")

                if pic_ci2_val is not None and np.isfinite(pic_ci2_val):
                    parts.append(f"PIC (CI2 - PACE granule): {pic_ci2_val:.3f} mmol/m³")
                
                if pic_dalec is not None and np.isfinite(pic_dalec):
                    parts.append(f"PIC (CI2 - DALEC): {pic_dalec:.3f} mmol/m³")

                if pic_ci2_rgb is not None and np.isfinite(pic_ci2_rgb):
                    parts.append(f"PIC (CI2 - PACE merged): {pic_ci2_rgb:.3f} mmol/m³")
                    
                if chla_dalec is not None and np.isfinite(chla_dalec):
                    parts.append(f"Chl A (DALEC, OC6): {chla_dalec:.3f} mg/m³")

                if chla_rgb is not None and np.isfinite(chla_rgb):
                    parts.append(f"Chl A (RGB, OC6): {chla_rgb:.3f} mg/m³")

                conc_text = "\n".join(parts) if parts else ""

                ax.set_xlabel(xlabel)
                ax.set_ylabel(ylabel)
                ax.legend(loc = 'upper right')
                ax.text(
                    0.02, 0.08,          # bottom-left in axes coordinates
                    conc_text,
                    transform=ax.transAxes,
                    fontsize=10,
                    verticalalignment='bottom',
                    horizontalalignment='left',
                    bbox=dict(boxstyle="round", facecolor='none', edgecolor='grey', alpha=0.8),
                )
                ax.set_ylim(0, 0.025)
                        

                if xlim is not None:
                    ax.set_xlim(xlim)
                if xticks is not None:
                    ax.set_xticks(xticks)
                if ylim is not None:
                    ax.set_ylim(ylim)

                if title is not None:
                    ax.set_title(title)
                else:
                    ax.set_title(f"Station: {station}")

                if grid:
                    ax.grid(**grid_kwargs)

                if minor_ticks:
                    ax.xaxis.set_minor_locator(AutoMinorLocator())
                    ax.yaxis.set_minor_locator(AutoMinorLocator())

                if spines_top_right:
                    ax.spines["top"].set_visible(True)
                    ax.spines["right"].set_visible(True)

                plt.tight_layout()

                if save_dir is not None:
                    # For single mode, you might want to include station in filename
                    fig.savefig(f'{save_dir}/{station}_rrs_spectrum.png', dpi=save_kwargs.get("dpi", dpi), bbox_inches="tight")

                if show:
                    plt.show()
                else:
                    plt.close(fig)

            # Return last figure (or None if you prefer)
            return fig

        if src_rrs == 'granule':
            for lon, lat, station in zip(lons, lats, stations):

                tol = 1e-6
                mask = (
                    np.isclose(matchup_ds["lon_insitu"], lon, atol=tol) &
                    np.isclose(matchup_ds["lat_insitu"], lat, atol=tol)
                )
                subset = matchup_ds.loc[mask, "granule"]

                subset_rrs = matchup_ds.loc[
                                    (matchup_ds["lon_insitu"] == lon) & (matchup_ds["lat_insitu"] == lat),
                                    "Rrs_547_px4_raw"
                                ]



                if subset.empty or subset_rrs.empty or pd.isna(subset_rrs).all():
                    # Fallback: plot from ds (RGB-like)
                    print(f"Warning: No granule found for station {station}. Plotting from rgb instead.")

                    chla_rgb, pic_rgb = chl_pic_oc6_pace_merged(ds, lat, lon)

                    test = ds[rrs_variable].sel(
                        longitude=lon,
                        latitude=lat,
                        method="nearest",
                    )

                    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

                    ax.plot(
                        test[wavelength_variable].values,
                        test.values,
                        **line_kwargs,
                        label = 'from merged L2 (Cruise Period)'
                    )



                else:
                    # Plot from granule
                    granule_filepath = subset.iloc[0]

                    chla_granule, pic_granule = chl_pic_oc6_pace_granule(granule_filepath, lon, lat)
                    pic_granule = pic_granule * 1000

                    x_granule, y_granule, flags = pace_granule_plot_prep(granule_filepath, lon, lat)

                    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

                    ax.plot(
                        x_granule,
                        y_granule,
                        **line_kwargs,
                        label = 'matched L2 granule'
                    )

                # Common plotting setup (DALEC, labels, etc.)
                if dalec is not None:
                    x, y, index = xy_dalec(dalec, lat, lon)
                    ax.plot(
                        x,
                        y,
                        color="black",
                        linestyle="--",
                        label="DALEC (Ship-Based)",
                    )
                    chla_dalec = dalec['chla_mg_m3'][index]
                    pic_dalec = dalec['pic_mmol_m3'][index] * 1000
 
                    print(f'{chla_dalec} mg/m3 found from DALEC rrs')
                    print(f'{pic_dalec} mmol/m3 found from DALEC rrs')
                     
 
 
                 # PIC and Chl A values
 
                 
 
                pic = discrete_ds.loc[
                                     (discrete_ds["lon"] == lon) & (discrete_ds["lat"] == lat),
                                     "PIC_mmolC_m3"
                                 ]
 
                print(f'{pic} discrete found for {lon}, {lat}')
 
                pic_ci2 = matchup_ds.loc[
                                     (matchup_ds["lon_insitu"] == lon) & (matchup_ds["lat_insitu"] == lat),
                                     "ci2_pic"
                                 ]
                 
                print(f'{pic_ci2} discrete found for {lon}, {lat}')
 
 
                def to_scalar_or_none(x):
                     # Returns None if x is “missing”, else a float scalar
                    if isinstance(x, pd.Series):
                        if x.empty:
                            return None
                        val = x.iloc[0]
                        return None if pd.isna(val) else float(val)
                     # Assume scalar-like
                    if pd.isna(x):
                         return None
                    return float(x)
 
                chla_dalec = to_scalar_or_none(chla_dalec)
                chla_granule = to_scalar_or_none(chla_granule)
                pic_granule = to_scalar_or_none(pic_granule)
                pic_dalec = to_scalar_or_none(pic_dalec)
                #pic_ci2_rgb = to_scalar_or_none(pic_rgb)
                pic_val = to_scalar_or_none(pic)
                pic_ci2_val = to_scalar_or_none(pic_ci2)
                #chla_rgb = to_scalar_or_none(chla_rgb)
 
                parts = []
 
                if pic_val is not None and np.isfinite(pic_val):
                    parts.append(f"PIC (ICP, In-Situ): {pic_val:.3f} mmol/m³")
                 
                if pic_dalec is not None and np.isfinite(pic_dalec):
                    parts.append(f"PIC (CI2 - DALEC): {pic_dalec:.3f} mmol/m³")

                #if pic_granule is not None and np.isfinite(pic_granule):
                 #   parts.append(f"PIC (CI2 - PACE L2 granule): {pic_granule:.3f} mmol/m³")
 
                #if pic_ci2_rgb is not None and np.isfinite(pic_ci2_rgb):
                #    parts.append(f"PIC (CI2 - PACE merged): {pic_ci2_rgb:.3f} mmol/m³")
                     
                if chla_dalec is not None and np.isfinite(chla_dalec):
                    parts.append(f"Chl A (DALEC, OC6): {chla_dalec:.3f} mg/m³")
 
                #if chla_rgb is not None and np.isfinite(chla_rgb):
                 #   parts.append(f"Chl A (RGB, OC6): {chla_rgb:.3f} mg/m³")

                if chla_granule is not None and np.isfinite(chla_granule):
                    parts.append(f"Chl A (PACE granule, OC6): {chla_granule:.3f} mg/m³")
 
                conc_text = "\n".join(parts) if parts else ""                 

                ax.set_xlabel(xlabel)
                ax.set_ylabel(ylabel)
                ax.legend(loc = 'upper right')
                ax.text(
                                    0.02, 0.08,          # bottom-left in axes coordinates
                                    conc_text,
                                    transform=ax.transAxes,
                                    fontsize=10,
                                    verticalalignment='bottom',
                                    horizontalalignment='left',
                                    bbox=dict(boxstyle="round", facecolor='none', edgecolor='grey', alpha=0.8),
                                )

                if xlim is not None:
                    ax.set_xlim(xlim)
                if xticks is not None:
                    ax.set_xticks(xticks)
                if ylim is not None:
                    ax.set_ylim(ylim)

                if title is not None:
                    ax.set_title(title)
                else:
                    ax.set_title(f"Station: {station}")

                if grid:
                    ax.grid(**grid_kwargs)

                if minor_ticks:
                    ax.xaxis.set_minor_locator(AutoMinorLocator())
                    ax.yaxis.set_minor_locator(AutoMinorLocator())

                if spines_top_right:
                    ax.spines["top"].set_visible(True)
                    ax.spines["right"].set_visible(True)

                plt.tight_layout()

                if save_dir is not None:
                    fig.savefig(
                        f"{save_dir}/{station}_rrs_spectrum.png",
                        dpi=save_kwargs.get("dpi", dpi),
                        bbox_inches="tight",
                    )

                if show:
                    plt.show()
                else:
                    plt.close(fig)

            # Return last figure (or None if you prefer) – outside the loop
            return fig

    elif mode == "multi":
        # All spectra in one plot
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

        colors = plt.cm.get_cmap(cmap)(np.linspace(cmap_range[0], cmap_range[1], len(stations)))

        for i, (lon, lat, station) in enumerate(zip(lons, lats, stations)):
            test = ds[rrs_variable].sel(
                longitude=lon,
                latitude=lat,
                method="nearest",
            )

            ax.plot(
                test[wavelength_variable].values,
                test.values,
                color=colors[i],
                label=f"{station}",
                **line_kwargs,
            )

        if dalec is not None:
                # Prepare DALEC-specific line style # or ':', '-.', etc.

            for lon, lat in zip(lons, lats):  # adapt to your DALEC loop

                x, y = xy_dalec(dalec, lat, lon)

                ax.plot(
                        x,
                        y,
                        linestyle="--",          # explicit, in case line_kwargs doesn't set it
                        label="DALEC",           # if you want it in the legend
                    )

        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.legend(loc = 'upper right')

        if xlim is not None:
            ax.set_xlim(xlim)
        if xticks is not None:
            ax.set_xticks(xticks)
        if ylim is not None:
            ax.set_ylim(ylim)

        if title is not None:

            ax.set_title(title)

        if grid:
            ax.grid(**grid_kwargs)

        if minor_ticks:
            ax.xaxis.set_minor_locator(AutoMinorLocator())
            ax.yaxis.set_minor_locator(AutoMinorLocator())

        if spines_top_right:
            ax.spines["top"].set_visible(True)
            ax.spines["right"].set_visible(True)

        if len(stations) > 0:
            ax.legend(title=legend_title, loc="upper right", frameon=False)

        plt.tight_layout()

        if save_dir is not None:
            fig.savefig(f"{save_dir}{stations[0][0:10]}_rrs_spectra_all.png", dpi=save_kwargs.get("dpi", dpi), bbox_inches="tight")

        if show:
            plt.show()

        return fig

    else:
        raise ValueError("mode must be 'single' or 'multi'.")


    

    

In [ ]:


# DALEC processing

DALEC_paths = [SS20250601_DALEC_path, SS20260601_DALEC_path]
keys = ['SS202506', 'SS202606']

# dict 
DALEC_ds = {}


for DALEC_path, key in zip(DALEC_paths, keys): 

    df = openDALEC(DALEC_path)

    DALEC_ds[key] = df




In [ ]:
matchups_discrete_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/matchups_discrete/240min_2km/matchups_2024-05-21 23:56:15_2026-06-05 16:03:35.csv'
matchups_discrete = pd.read_csv(matchups_discrete_path)

matchups_discrete['ci2_pic'] = CL2algorithm(matchups_discrete['Rrs_547_px4_raw'], matchups_discrete['Rrs_667_px4_raw'], old = True) *1000


discrete = pd.read_csv(path_all_pic)


save_dir = '/Volumes/Extreme SSD/Bloom 202606/validation_plots_granule'
ds1 = xr.open_dataset(SS20260601_allwls)
ds2 = xr.open_dataset(SS20250601_allwls)
dalec1, dalec2 = DALEC_ds['SS202606'], DALEC_ds['SS202506']

df1 = SS_campaigns['SS20260601']
df2 = SS_campaigns['SS20250602']



dfs = [df2]
dalecs = [dalec2]
dss = [ds2]

modes = ["single"]


for mode in modes:
        
        for df, dalec, ds in  zip(dfs, dalecs, dss):

            lons = list(df["lon"])
            lats = list(df["lat"])
            stations = list(df["Station"])
            datetimes = list(df["datetime"])

            plot_rrs_spectra(
                ds = ds,
                lons= lons,
                lats= lats,
                stations= stations,
                rrs_variable="Rrs",
                wavelength_variable = "wavelength",
                mode = mode,  # "single" or "multi"
                figsize=  (6.5, 4.5),
                dpi = 300,
                xlim= (400, 700),
                ylim = None,
                xticks= None,
                xlabel = "Wavelength (nm)",
                ylabel = r"Remote-sensing reflectance, $R_{rs}$ (sr$^{-1}$)",
                title = None,
                legend_title = "Station",
                cmap= "viridis",
                cmap_range = (0, 0.9),
                line_kwargs = None,
                grid = True,
                grid_kwargs= None,
                minor_ticks = True,
                spines_top_right = True,
                save_dir = save_dir,
                save_kwargs = None,
                show= True,
                apply_style = True,
                dalec = dalec,
                src_rrs= 'rgb',
                matchup_ds = matchups_discrete,
                discrete_ds = discrete
                )